# 自定义全局状态

向状态添加额外的字段，以定义复杂的行为，而无需依赖消息列表

# 1、向State添加字段并在工具内部更新State

通过向状态添加 name 和 birthday 键来更新聊天机器人，以研究实体的生日。

在 human_assistance 工具内部填充状态键。这允许人工在信息存储到状态之前对其进行审查。使用 Command 从工具内部发出状态更新

In [1]:
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
import os
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool, InjectedToolCallId
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import ToolNode, tools_condition

from langgraph.types import Command, interrupt

from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

# 加载 .env 文件
load_dotenv(override=True)


os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "langgraph_learnning"
os.environ["LANGSMITH_ENDPOINT"] = "https://aws.api.smith.langchain.com"

############### 全局状态新增name和birthday两个字段 ###############
class State(TypedDict):
    messages: Annotated[list, add_messages]
    name: str
    birthday: str
######################################################
    
graph_builder = StateGraph(State)

search_tool = TavilySearch(max_results=2,
                    tavily_api_key=os.getenv("TAVILY_API_KEY"))

############### 全局状态新增name和birthday两个字段 ###############
@tool
# 这里要注意，我们这段代码的目的是生成一条 ToolMessage 来更新 State，而创建 ToolMessage
# 就必须知道这次工具调用的 ID。但这个 ID 不该让大模型自己来填，所以用 InjectedToolCallId
# 标记一下，框架会自动注入，大模型在工具的参数描述里看不到它
def human_assistance(
    name: str, birthday: str, tool_call_id: Annotated[str, InjectedToolCallId]
) -> str:
    """请求人工协助"""
    human_response = interrupt(
        {
            "question": "Is this correct?",
            "name": name,
            "birthday": birthday,
        },
    )
    # If the information is correct, update the state as-is.
    if human_response.get("correct", "").lower().startswith("y"):
        verified_name = name
        verified_birthday = birthday
        response = "Correct"
    # Otherwise, receive information from the human reviewer.
    else:
        verified_name = human_response.get("name", name)
        verified_birthday = human_response.get("birthday", birthday)
        response = f"Made a correction: {human_response}"

    # This time we explicitly update the state with a ToolMessage inside
    # the tool.
    state_update = {
        "name": verified_name,
        "birthday": verified_birthday,
        "messages": [ToolMessage(response, tool_call_id=tool_call_id)],
    }
    # We return a Command object in the tool to update our state.
    return Command(update=state_update)
######################################################

tools = [search_tool, human_assistance]

llm = ChatOpenAI(
    model="mimo-v2-pro",
    api_key=os.getenv("XIAOMI_API_KEY"),
    base_url=os.getenv("XIAOMI_BASE_URL"),
)

# llm 绑定上工具
llm_with_tool = llm.bind_tools(tools)

def chatbot(state: State):
    return {"messages": [llm_with_tool.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)



tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)

# 每次调用工具后，都会回到聊天机器人来决定下一步
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")
graph = graph_builder.compile(checkpointer=memory)

# 2、提示聊天机器人

提示聊天机器人查找 LangGraph 库的“生日”，并指示聊天机器人一旦获得所需信息就使用 human_assistance 工具。通过在工具参数中设置 name 和 birthday，可以强制聊天机器人为这些字段生成建议

In [2]:
user_input = (
    "你能查一下 LangGraph 是什么时候发布的吗？"
    "查到答案后，用 human_assistance 工具来让人工审核一下。"
)
config = {"configurable": {"thread_id": "1"}}

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

你能查一下 LangGraph 是什么时候发布的吗？查到答案后，用 human_assistance 工具来让人工审核一下。
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_0c2e3c852e074b8b821176cb)
 Call ID: call_0c2e3c852e074b8b821176cb
  Args:
    query: LangGraph 发布时间
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search

{"query": "LangGraph 发布时间", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.reddit.com/r/LangChain/comments/1fo4mp4/langgraph_studio_release_timeline/?tl=zh-hans", "title": "LangGraph Studio 发布时间表: r/LangChain", "content": "LangGraph Studio 目前只对Apple Silicon 芯片开放Beta 测试，那么它什么时候会发布到其他操作系统平台，比如Windows 和Linux 呢？", "score": 0.9999138, "raw_content": null}, {"url": "https://zhuanlan.zhihu.com/p/679545767", "title": "LangChain首个稳定版本终于发布，LangGraph把智能体 ...", "content": "奋战一

**注意**: 这里进入了interrupt，等待用户的输入

# 3、添加人工协助

聊天机器人未能识别正确的日期，因此为其提供信息

In [3]:
human_command = Command(
    resume={
        "name": "LangGraph",
        "birthday": "Jan 17, 2024",
    },
)

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_db42e08e01f04807afad47f6)
 Call ID: call_db42e08e01f04807afad47f6
  Args:
    name: LangGraph 发布时间
    birthday: 2024-01-22
================================= Tool Message =================================
Name: human_assistance

Made a correction: {'name': 'LangGraph', 'birthday': 'Jan 17, 2024'}
================================== Ai Message ==================================

根据我的查询和人工审核确认，**LangGraph 的正式发布日期是 2024 年 1 月 17 日**。

这个信息来自 LangChain 官方更新日志，其中明确显示 "Introducing LangGraph" 的公告发布于 2024 年 1 月 22 日那一周。经过人工审核工具进一步确认，更精确的发布日期是 2024 年 1 月 17 日。

LangGraph 是 LangChain 团队推出的一个重要框架，用于构建有状态、可循环的多智能体应用程序，它扩展了 LangChain 的功能，特别适合创建需要复杂工作流和状态管理的 AI 应用。


In [4]:
snapshot = graph.get_state(config)

{k: v for k, v in snapshot.values.items() if k in ("name", "birthday")}

{'name': 'LangGraph', 'birthday': 'Jan 17, 2024'}

# 4、手动更新状态

LangGraph 对应用程序状态提供高度控制。例如，在**任何时候（包括中断时）**，您都可以使用 graph.update_state 手动覆盖一个键

In [5]:
graph.update_state(config, {"name": "LangGraph (library)"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f14862a-c1b3-6649-801c-4c07c6a330de'}}

In [7]:
snapshot = graph.get_state(config)

{k: v for k, v in snapshot.values.items() if k in ("name", "birthday")}

{'name': 'LangGraph (library)', 'birthday': 'Jan 17, 2024'}